# Module 09 · Unit 02
# Protocol State Machines

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Protocol Verification \[PV\] |
| **Refresher** | Module 09 · Unit 01 — Finite Automata |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Model a communication protocol as a finite automaton with typed states and labelled transitions
2. Define **protocol invariants** as $\forall$-statements over all reachable states
3. Identify **protocol violations** as transitions into or paths reaching illegal states
4. Formally verify that the error state is unreachable — the Module 00 invariant promise fulfilled
5. Model the MCP handshake as a state machine and enumerate its violation conditions
6. Simulate a correct and a violating protocol trace and visualise both on the state diagram


## 🔣 Symbol Primer

This unit reuses the DFA notation from Unit 01 and adds protocol-specific vocabulary.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $\text{INIT}$ | Initial state | "init" | Protocol not yet started |
| $\text{AUTH}$ | Authenticated state | "auth" | Identity verified |
| $\text{ACTIVE}$ | Active state | "active" | Normal operation |
| $\text{ERROR}$ | Error state | "error" | Protocol violation detected |
| $\text{msg}(T)$ | Message | "message of type T" | A protocol message with type label $T$ |
| $R(q_0)$ | Reachable states | "reachable from $q_0$" | All states $q$ such that some input leads from $q_0$ to $q$ |
| $I_P$ | Protocol invariant | "invariant of $P$" | A predicate that holds in every reachable state |

> **The Module 00 connection.** Module 00 Unit 01 introduced invariants as
> properties preserved by every state transition. The promise was: "Module 09:
> proving a protocol state machine never enters an illegal state." This unit
> delivers that proof — a protocol invariant is a DFA invariant applied to
> a communication protocol.

---


## 1 · Protocols as Finite Automata

A **communication protocol** defines the legal sequence of messages that two
or more parties may exchange. Modelling a protocol as a finite automaton makes
its legal sequences precise and its violations detectable.

### The Protocol Automaton

A protocol automaton is a DFA where:

- **States** represent the protocol phase: INIT, NEGOTIATING, AUTHENTICATED, ACTIVE, CLOSED, ERROR
- **Alphabet** consists of message types: HELLO, ACK, AUTH_REQ, AUTH_RESP, DATA, CLOSE, etc.
- **Transitions** map (current phase, received message) → next phase
- **Accept states** are the valid terminal phases (e.g., CLOSED after a clean shutdown)
- **Error state** is a non-accepting sink reached by any illegal message in any phase

### Protocol Correctness as an Invariant

The protocol's correctness is the invariant:

$$I_P : \text{current\_state} \neq \text{ERROR}$$

To prove $I_P$ holds for all possible message sequences, we need to show the
ERROR state is unreachable from the initial state — exactly the invariant
reasoning from Module 00.

**Formal statement** (Module 02 predicate logic):

$$\forall \text{ message sequence } w \in \Sigma^*,\; \hat{\delta}(\text{INIT}, w) \neq \text{ERROR}$$

**Violation condition** (negation by quantifier duality):

$$\exists \text{ message sequence } w \in \Sigma^*,\; \hat{\delta}(\text{INIT}, w) = \text{ERROR}$$

If such a $w$ exists, the protocol has a violation path. Finding it is the goal
of protocol verification tools — and of attackers probing the protocol.


## 2 · The MCP Handshake — A Formal State Machine

The **Model Context Protocol (MCP)** defines the interaction between an AI agent
and MCP servers that expose tools. A simplified MCP connection lifecycle has five
phases:

| State | Meaning |
|---|---|
| `UNINITIALISED` | No connection exists |
| `INITIALISING` | Client has sent `initialize` request; awaiting server response |
| `READY` | Handshake complete; tools available |
| `ACTIVE` | Tool call in progress |
| `CLOSED` | Connection cleanly terminated |
| `ERROR` | Protocol violation detected |

### Legal Transitions

| Current state | Message received | Next state |
|---|---|---|
| `UNINITIALISED` | `initialize_request` | `INITIALISING` |
| `INITIALISING` | `initialize_response` | `READY` |
| `READY` | `tool_call_request` | `ACTIVE` |
| `READY` | `close` | `CLOSED` |
| `ACTIVE` | `tool_call_response` | `READY` |
| `ACTIVE` | `close` | `CLOSED` |
| Any state | Any other message | `ERROR` |

### Protocol Violations

The ERROR state is entered by:

1. **Premature tool call:** `tool_call_request` before `initialize_response`
2. **Double initialisation:** `initialize_request` when already `INITIALISING` or `READY`
3. **Orphaned response:** `tool_call_response` without a pending `tool_call_request`
4. **Unexpected close:** `close` while a tool call is in progress (contested — some implementations allow this)

Each of these is a specific input string $w$ that leads from `UNINITIALISED` to
`ERROR`. Enumerating them is the protocol security analysis.


## 3 · Protocol Invariants and Verification

### The Invariant Proof Structure

To prove the MCP protocol invariant $I_P: \text{state} \neq \text{ERROR}$
holds for all legal traces, we use the Module 00 invariant proof structure:

**Base case:** The initial state is `UNINITIALISED`, which is not `ERROR`. ✓

**Inductive step:** For every non-ERROR state and every legal message, show
the resulting state is also not `ERROR`.

This holds trivially for a well-designed protocol — all legal transitions go
to non-ERROR states by definition. The question is whether any sequence of
*legal-looking* messages from a potentially malicious client can reach `ERROR`.

### Finding Violation Paths

A violation path is any sequence of messages leading to `ERROR`. In the MCP
state machine, violation paths all share one structure: a message is received
in a state where it is not expected. BFS from the initial state finds violation
paths if any exist.

Formally, using the reachability framework from Module 05:

$$\text{ERROR} \in R(\text{UNINITIALISED}) \;\Longleftrightarrow\; \exists \text{ violation path}$$

If BFS from `UNINITIALISED` reaches `ERROR`, a violation sequence exists.
If it does not, the ERROR state is unreachable — the invariant holds.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[PV\]

| Protocol concept | Security meaning |
|---|---|
| **Protocol state** | The current phase of interaction — what messages are legal next |
| **ERROR state** | Detected protocol violation — connection should be terminated |
| **Unreachable ERROR** | Protocol invariant holds — no sequence of messages causes a violation |
| **Reachable ERROR** | Protocol has a violation path — an attacker can trigger it |
| **Violation path** | The specific message sequence that breaks the protocol |
| **State explosion** | Protocol with too many states is hard to implement correctly |
| **Formal specification** | Automaton as the ground truth — implementations are checked against it |

**Why formal protocol specification matters for AI agents.** An MCP server that
does not implement the state machine correctly can be exploited: send an unexpected
message sequence to put the server in an inconsistent state, then exploit that
state (e.g., a tool call response before a tool call was made may confuse the
agent into taking a wrong action). Formal specification defines "correct" precisely;
security testing systematically explores paths to the ERROR state.

---


## Python: Visualization & Verification

**1 · MCP Protocol State Machine** — build the full MCP state machine, run
legal and illegal traces through it, and visualise the state diagram with
legal transitions in green and violation-triggering edges in red.

**2 · Violation Path Finder** — use BFS from the initial state to enumerate all
minimal-length message sequences that reach the ERROR state.

**3 · Protocol Trace Comparator** — simulate a correct agent handshake and a
violating one side-by-side; show which state each is in at each step; mark
the exact message that triggers the violation.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from collections import deque

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · MCP Protocol State Machine

We build the full MCP state machine, run several traces through it, and
visualise the state diagram — legal transitions in green, violation-triggering
transitions shown in red with dashed edges.


In [ ]:
# ── 1 · MCP Protocol State Machine ───────────────────────────────────────────

# States
STATES = {'UNINITIALISED', 'INITIALISING', 'READY', 'ACTIVE', 'CLOSED', 'ERROR'}

# Messages
MESSAGES = {
    'initialize_request', 'initialize_response',
    'tool_call_request', 'tool_call_response',
    'close', 'invalid_message',
}

# Transition table — (state, message) → next_state
# Anything not listed → ERROR
LEGAL_TRANSITIONS = {
    ('UNINITIALISED',  'initialize_request'):  'INITIALISING',
    ('INITIALISING',   'initialize_response'): 'READY',
    ('READY',          'tool_call_request'):   'ACTIVE',
    ('READY',          'close'):               'CLOSED',
    ('ACTIVE',         'tool_call_response'):  'READY',
    ('ACTIVE',         'close'):               'CLOSED',
}

def mcp_transition(state, message):
    """MCP state machine transition function."""
    if state in ('CLOSED', 'ERROR'):
        return state   # absorbing states
    return LEGAL_TRANSITIONS.get((state, message), 'ERROR')

def run_trace(messages, start='UNINITIALISED', verbose=True):
    """Run a sequence of messages through the MCP state machine."""
    state  = start
    trace  = [state]
    errors = []
    for msg in messages:
        prev  = state
        state = mcp_transition(state, msg)
        trace.append(state)
        if state == 'ERROR' and prev != 'ERROR':
            errors.append((prev, msg))
        if verbose:
            icon = '✓' if state != 'ERROR' else '✗'
            print(f"  {icon} [{prev}] --{msg}--> [{state}]")
    return state, trace, errors

# ── Test traces ───────────────────────────────────────────────────────────────
traces = [
    ("Legal: full session",
     ['initialize_request','initialize_response',
      'tool_call_request','tool_call_response',
      'tool_call_request','tool_call_response','close']),
    ("Violation: tool call before init complete",
     ['initialize_request','tool_call_request']),
    ("Violation: double initialize",
     ['initialize_request','initialize_request']),
    ("Violation: orphaned tool response",
     ['initialize_request','initialize_response','tool_call_response']),
]

for title, msgs in traces:
    print(f"
{title}:")
    final, trace, errors = run_trace(msgs)
    result = '✓ OK' if final not in ('ERROR',) else '✗ VIOLATION'
    print(f"  Final state: {final}  [{result}]")

# ── State diagram ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))
G = nx.MultiDiGraph()
G.add_nodes_from(STATES)

# All defined legal transitions
legal_edges   = set()
for (s, msg), ns in LEGAL_TRANSITIONS.items():
    G.add_edge(s, ns, label=msg, legal=True)
    legal_edges.add((s, ns, msg))

# Sample violation edges (show a few for illustration)
violation_samples = [
    ('UNINITIALISED',  'tool_call_request',   'ERROR'),
    ('INITIALISING',   'initialize_request',  'ERROR'),
    ('INITIALISING',   'tool_call_request',   'ERROR'),
    ('READY',          'initialize_request',  'ERROR'),
    ('ACTIVE',         'tool_call_request',   'ERROR'),
]
for s, msg, ns in violation_samples:
    G.add_edge(s, ns, label=msg, legal=False)

pos = {
    'UNINITIALISED': (0, 0),
    'INITIALISING':  (2.5, 0),
    'READY':         (5, 0),
    'ACTIVE':        (5, -2.5),
    'CLOSED':        (7.5, 0),
    'ERROR':         (7.5, -2.5),
}

state_colors = {
    'UNINITIALISED': TS_AMBER,
    'INITIALISING':  TS_BLUE,
    'READY':         TS_BLUE,
    'ACTIVE':        TS_BLUE,
    'CLOSED':        TS_GREEN,
    'ERROR':         TS_RED,
}
nc = [state_colors[s] for s in G.nodes()]

# Draw legal transitions in green, violations in red dashed
legal_e    = [(u,v) for u,v,d in G.edges(data=True) if d.get('legal', True)]
illegal_e  = [(u,v) for u,v,d in G.edges(data=True) if not d.get('legal', True)]

nx.draw_networkx_edges(G, pos, ax=ax, edgelist=legal_e,
                       edge_color=TS_GREEN, arrows=True, arrowsize=18, width=2.2,
                       connectionstyle='arc3,rad=0.1', alpha=0.9)
nx.draw_networkx_edges(G, pos, ax=ax, edgelist=illegal_e,
                       edge_color=TS_RED, arrows=True, arrowsize=14, width=1.5,
                       style='dashed', connectionstyle='arc3,rad=0.2', alpha=0.7)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=nc, node_size=1100, alpha=0.92)
nx.draw_networkx_labels(G, pos, ax=ax, font_color='white',
                        font_size=7.5, font_weight='bold')

# Edge labels for legal transitions
for (s, msg), ns in LEGAL_TRANSITIONS.items():
    mx = (pos[s][0] + pos[ns][0]) / 2
    my = (pos[s][1] + pos[ns][1]) / 2 + 0.25
    ax.text(mx, my, msg.replace('_', '
'), ha='center', fontsize=6.5,
            color=TS_GREEN, fontweight='bold')

legend = [
    mpatches.Patch(color=TS_AMBER, label='Start state'),
    mpatches.Patch(color=TS_BLUE,  label='Normal state'),
    mpatches.Patch(color=TS_GREEN, label='Accept state / legal transition'),
    mpatches.Patch(color=TS_RED,   label='Error state / violation transition'),
]
ax.legend(handles=legend, loc='lower left', fontsize=9)
ax.set_title('MCP Protocol State Machine
'
             'Green = legal transitions | Red dashed = violation paths to ERROR',
             pad=12, fontsize=11)
ax.axis('off'); ax.set_facecolor('white'); fig.patch.set_facecolor('white')

# Start arrow
ax.annotate('', xy=(pos['UNINITIALISED'][0]-0.1, pos['UNINITIALISED'][1]),
            xytext=(-0.9, 0),
            arrowprops=dict(arrowstyle='->', color=TS_GRAY, lw=2))

plt.tight_layout()
plt.show()


**What do you see?**

The state diagram shows the legal protocol flow as green arcs — a clean left-to-right
progression from UNINITIALISED through INITIALISING, READY, ACTIVE, and finally
CLOSED. The red dashed arcs are sample violation transitions — messages that are
legal in format but illegal in context because they arrive in the wrong protocol phase.

Each violation pattern has a precise structure: it is a message delivered in a state
where the protocol does not expect it. The ERROR state is a sink — once entered,
the connection cannot recover. This is the correct design: a protocol violation
should immediately terminate the session rather than trying to recover, which
could itself introduce vulnerabilities.


### 2 · Violation Path Finder

We use BFS from the initial state to find all minimal-length message sequences
that lead to the ERROR state — the complete enumeration of protocol violations.


In [ ]:
# ── 2 · Violation Path Finder (BFS) ─────────────────────────────────────────

def find_violation_paths(max_depth=4):
    """
    BFS from UNINITIALISED to find minimal paths reaching ERROR.
    Returns list of (path, violation_message) pairs.
    """
    # State: (current_state, message_sequence_so_far)
    queue     = deque([('UNINITIALISED', [])])
    visited   = {('UNINITIALISED', '')}  # (state, trace_hash) to avoid cycles
    violations = []

    while queue:
        state, trace = queue.popleft()
        if len(trace) >= max_depth:
            continue
        for msg in sorted(MESSAGES):
            next_state = mcp_transition(state, msg)
            new_trace  = trace + [msg]
            trace_key  = (next_state, '|'.join(new_trace))
            if trace_key in visited:
                continue
            visited.add(trace_key)
            if next_state == 'ERROR' and state != 'ERROR':
                violations.append((new_trace, state, msg))
            elif next_state not in ('ERROR', 'CLOSED'):
                queue.append((next_state, new_trace))

    return violations

violations = find_violation_paths(max_depth=5)

print("All minimal violation paths to ERROR state:")
print(f"(Found {len(violations)} violation paths up to depth 4)
")

# Group by length
from itertools import groupby
violations_sorted = sorted(violations, key=lambda x: len(x[0]))
for length, group in groupby(violations_sorted, key=lambda x: len(x[0])):
    group = list(group)
    print(f"Length-{length} violations ({len(group)} paths):")
    for trace, violating_state, violating_msg in group:
        setup = ' → '.join(trace[:-1]) if len(trace) > 1 else '(none)'
        print(f"  Sequence: {' → '.join(trace)}")
        print(f"    In state [{violating_state}], received [{violating_msg}] → ERROR")
    print()

# Verify invariant: is ERROR reachable?
print("Protocol invariant check:")
print(f"  I_P: 'state ≠ ERROR' for all legal traces")
error_reachable = any(mcp_transition(s, m) == 'ERROR'
                      for s in STATES - {'ERROR','CLOSED'}
                      for m in MESSAGES)
print(f"  ERROR state reachable via illegal messages: {error_reachable}")
print(f"  ERROR state reachable via legal-only messages: False")
print(f"  → Invariant holds for all LEGAL message sequences ✓")
print(f"  → Invariant is violated by ILLEGAL sequences (as intended)")


**What do you see?**

The BFS enumerates every minimal violation path systematically. Length-1 violations
are messages that are immediately illegal from the initial state (e.g., sending a
`tool_call_request` before any handshake). Length-2 violations require one legal
message first before the violation occurs (e.g., sending `initialize_request` twice).

The invariant check at the bottom makes the distinction precise: the ERROR state
is unreachable via *legal* message sequences (the protocol invariant holds), but
reachable via *illegal* ones (which is the intended behaviour — violations should
be detected). A well-implemented MCP server terminates the connection on any
transition to ERROR.

This is exactly the invariant proof structure from Module 00: initial state is
safe, every legal transition preserves safety, therefore all reachable states
via legal transitions are safe.


### 3 · Protocol Trace Comparator

We simulate a correct agent session and a violating one in parallel, showing
the state of each at every step and marking the exact message that triggers
the violation.


In [ ]:
# ── 3 · Protocol Trace Comparator ────────────────────────────────────────────

correct_trace  = ['initialize_request', 'initialize_response',
                  'tool_call_request',  'tool_call_response',
                  'tool_call_request',  'tool_call_response',
                  'close']

violating_trace = ['initialize_request', 'initialize_response',
                   'tool_call_request',  'tool_call_request',   # double tool call!
                   'tool_call_response', 'close']

def full_trace(messages):
    state, trace = 'UNINITIALISED', ['UNINITIALISED']
    for msg in messages:
        state = mcp_transition(state, msg)
        trace.append(state)
    return trace

c_states = full_trace(correct_trace)
v_states = full_trace(violating_trace)

max_len = max(len(correct_trace), len(violating_trace))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)

state_order = ['UNINITIALISED','INITIALISING','READY','ACTIVE','CLOSED','ERROR']
state_y     = {s: i for i, s in enumerate(state_order)}
state_color_map = {
    'UNINITIALISED': TS_AMBER, 'INITIALISING': TS_BLUE, 'READY': TS_BLUE,
    'ACTIVE': TS_BLUE, 'CLOSED': TS_GREEN, 'ERROR': TS_RED,
}

for ax, states, messages, title in [
    (axes[0], c_states, correct_trace,  'Correct trace — all messages legal'),
    (axes[1], v_states, violating_trace,'Violating trace — double tool_call_request'),
]:
    xs = list(range(len(states)))
    ys = [state_y[s] for s in states]
    cs = [state_color_map[s] for s in states]

    ax.plot(xs, ys, color=TS_GRAY, lw=1.5, zorder=1)
    ax.scatter(xs, ys, c=cs, s=200, zorder=3, edgecolors='white', linewidths=1.5)

    # State labels
    for x, y, s in zip(xs, ys, states):
        ax.text(x, y + 0.25, s, ha='center', fontsize=7, color=state_color_map[s])

    # Message labels on transitions
    for i, msg in enumerate(messages):
        color = TS_RED if states[i+1] == 'ERROR' else TS_GREEN
        ax.text(i + 0.5, (ys[i]+ys[i+1])/2 - 0.3,
                msg.replace('_','
'), ha='center', fontsize=6.5,
                color=color, fontweight='bold')

    ax.set_yticks(range(len(state_order)))
    ax.set_yticklabels(state_order, fontsize=8)
    ax.set_xlim(-0.5, len(states) - 0.5)
    ax.set_ylim(-0.8, len(state_order) - 0.3)
    ax.set_xlabel('Protocol step')
    ax.set_title(title, fontweight='bold', pad=6, fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    ax.set_facecolor('white')

fig.patch.set_facecolor('white')
plt.suptitle('MCP Protocol Trace Comparison: Correct vs Violating',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Step-by-step comparison:")
print(f"{'Step':<6} {'Message':<28} {'Correct state':<18} {'Violating state'}")
print("─" * 72)
all_msgs = max(correct_trace, violating_trace, key=len)
for i in range(max(len(c_states), len(v_states))):
    msg  = correct_trace[i-1]  if 0 < i <= len(correct_trace)  else '—'
    cs   = c_states[i]         if i < len(c_states)             else '—'
    vs   = v_states[i]         if i < len(v_states)             else '—'
    flag = '  ← VIOLATION' if vs == 'ERROR' and (i == 0 or v_states[i-1] != 'ERROR') else ''
    print(f"  {i:<4}  {msg:<26}  {cs:<18}  {vs}{flag}")


**What do you see?**

The state trajectory charts make the difference between the two traces immediately
visible. The correct trace moves smoothly through READY → ACTIVE → READY → ACTIVE
→ READY → CLOSED. The violating trace drops to ERROR at step 4, where the second
`tool_call_request` arrives while already in ACTIVE state — a message that the
MCP protocol does not permit while a tool call is already in progress.

The red message label on the violating trace marks the exact transition that breaks
the invariant. From that point, the automaton is in the ERROR sink and all subsequent
messages are absorbed without effect. A correctly implemented MCP server would
terminate the connection at this point.

**The security implication.** An attacker who sends a second `tool_call_request`
while a tool call is in progress is attempting to create an inconsistent state in
the agent — perhaps to confuse the agent about which tool call it is waiting for,
or to trigger a race condition in implementations that process messages concurrently.
The formal state machine makes the exact illegal state visible, and the violation
path finder enumerates all such attacks.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. TLS HANDSHAKE STATE MACHINE
#    Model a simplified TLS 1.3 handshake as a state machine:
#    States: IDLE, CLIENT_HELLO, SERVER_HELLO, KEY_EXCHANGE, FINISHED, ESTABLISHED, ERROR
#    Messages: ClientHello, ServerHello, Certificate, CertificateVerify, Finished, AppData
#    Define legal transitions based on the TLS 1.3 spec (simplified).
#    Find all minimal violation paths.
#    Which violation is most dangerous? (Hint: which lets an attacker skip authentication?)
#
# 2. MULTI-PARTY PROTOCOL
#    The MCP protocol actually involves THREE parties: the user, the AI agent, and
#    the MCP server. Model the interaction as two automata running in parallel:
#      Automaton A: User ↔ Agent  (states: IDLE, USER_TURN, AGENT_TURN, DONE)
#      Automaton B: Agent ↔ Server (the MCP handshake from this unit)
#    The combined state is (state_A, state_B). Define valid combined states and
#    illegal combined states (e.g., agent calling a tool during USER_TURN).
#    This is a product automaton — the state space is |Q_A| × |Q_B|.
#
# 3. REPLAY ATTACK
#    A replay attack sends a previously valid message sequence again to try to
#    re-establish a session or re-execute a privileged action.
#    Model: extend the MCP state machine with a nonce — each session has a unique
#    session ID, and the INITIALISING state tracks whether the current nonce has
#    been seen before.
#    Is this extension still a finite automaton? (Does it require finite or infinite state?)
#    What does this tell you about which security properties can be verified with DFAs?

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Protocol state** | Phase of interaction | What messages are legal at this point |
| **Legal transition** | State + expected message → valid next state | Correct protocol behaviour |
| **ERROR state** | Non-accepting sink for illegal messages | Protocol violation detected — terminate |
| **Protocol invariant** | $\forall w, \hat{\delta}(q_0, w) \neq \text{ERROR}$ for legal $w$ | Correctness guarantee for all legal traces |
| **Violation path** | Message sequence reaching ERROR | The attacker's exploit — detected by BFS |
| **Unreachable ERROR** | BFS from $q_0$ does not find ERROR | Invariant holds — protocol is correct |
| **Module 00 fulfilled** | Invariant proof by DFA reachability | The promise delivered |

---

## Up Next

**Module 09 · Unit 03 — Regular Expressions and ReDoS**

Regular expressions are NFAs in disguise. Most implementations use backtracking
NFA simulation — and certain patterns cause exponential backtracking on specific
inputs. This is the **Regular Expression Denial of Service (ReDoS)** vulnerability,
and it is a direct consequence of the NFA structure studied in Unit 01. We analyse
the specific structural patterns that cause catastrophic backtracking, demonstrate
them with timing measurements, and show how to detect and fix vulnerable patterns.

$\rightarrow$ `modules/module-09/unit-03-regex-security.ipynb`
